In [1]:
# Cell 1: تثبيت المكتبات المطلوبة لعملية التعلم الآلي ومعالجة النصوص
!pip install scikit-learn nltk pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [9]:
# Cell 2: المكتبات الي هنستخدمها 
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pickle

nltk.download('stopwords', quiet=True)
print("Libraries loaded successfully ")

Libraries loaded successfully 


In [3]:
# Cell 3: تحميل الداتا وتنظيف النصوص (إزالة الروابط والأرقام والكلمات المشتركة)
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words and len(word) > 1]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

print("Loading dataset...")
df = pd.read_csv("spam.csv", encoding='latin-1')
df = df[['v1', 'v2']]
df.columns = ['label', 'message']
df['label_encoded'] = df['label'].map({'ham': 0, 'spam': 1})
df['cleaned'] = df['message'].apply(preprocess_text)

print(f"Dataset loaded: {df.shape[0]} messages")
print(f"Spam: {(df['label'] == 'spam').sum()}")
print(f"Ham:  {(df['label'] == 'ham').sum()}")

Loading dataset...
Dataset loaded: 5572 messages
Spam: 747
Ham:  4825


In [4]:
# Cell 4: تقسيم الداتا، تدريب موديل SVM، واختبار دقته
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned'], df['label_encoded'],
    test_size=0.2, random_state=42, stratify=df['label_encoded']
)

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

svm_model = LinearSVC(C=1.0, max_iter=10000, class_weight='balanced', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

y_pred = svm_model.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print(f"ACCURACY: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

ACCURACY: 98.21%

Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       966
        Spam       0.96      0.91      0.93       149

    accuracy                           0.98      1115
   macro avg       0.97      0.95      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [10]:
# Cell 5: حفظ الموديل والـ Vectorizer كملفات pkl لاستخدامهم لاحقاً في الـ GUI
with open("svm_model.pkl", 'wb') as f:
    pickle.dump(svm_model, f)
print("svm_model.pkl saved! ")

with open("svm_vectorizer.pkl", 'wb') as f:
    pickle.dump(vectorizer, f)
print("svm_vectorizer.pkl saved! ")

svm_model.pkl saved! 
svm_vectorizer.pkl saved! 


In [11]:
# Cell 6: تجربة الموديل يدوياً بإدخال أي رسالة لمعرفة هل هي سبام ولا لا
import pickle

# 1. تحميل الموديل والـ Vectorizer اللي حفظناهم في الخطوة السابقة
with open("svm_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)
with open("svm_vectorizer.pkl", "rb") as f:
    loaded_vectorizer = pickle.load(f)

# 2. عمل دالة تأخذ الرسالة وترجع النتيجة
def test_message(text):
    cleaned = preprocess_text(text) # بنستخدم دالة التنظيف من الخلية 3
    vec = loaded_vectorizer.transform([cleaned])
    prediction = loaded_model.predict(vec)[0]
    if prediction == 1:
        return "Spam"
    else:
        return "Not Spam"

# 3. جربي تغيير الرسالة دي لأي رسالة عايزتها
my_msg = "Congratulations! You've won a FREE $1000 gift card."
print(f"Message: {my_msg}")
print(f"Result:  {test_message(my_msg)}")

Message: Congratulations! You've won a FREE $1000 gift card.
Result:  Spam


In [14]:
my_msg1 = "to reset your password, click the link below and follow the instructions"
print(f"Message: {my_msg1}")
print(f"Result:  {test_message(my_msg1)}")

Message: to reset your password, click the link below and follow the instructions
Result:  Spam


In [15]:
my_msg2 = "Win a FREE vacation to Dubai! Just enter your details: wintrip-now.com"
print(f"Message: {my_msg2}")
print(f"Result:  {test_message(my_msg2)}")

Message: Win a FREE vacation to Dubai! Just enter your details: wintrip-now.com
Result:  Spam
